ReAct = Reasoning + Acting. Agents alternate between reasoning about what it needs and taking actions to get it.  If the first search returned unhelpful results, the agent could try a different query. If the calculation seemed wrong, it could verify by searching for additional information.
Running an agent requires tracking:
- events: The history of all interactions, including user messages, LLMresponses, and tool execution results 
- current_step: A counter to prevent infinite loops (what if the agent
keeps calling tools forever?) 
- execution_id: A unique identifier for debugging and logging (which
execution produced this error?) 
- state: A scratchpad for dynamic data during execution, such as task
progress, intermediate results, or user preferences that tools might need 
- final_result: A flag to know when execution is complete

Consolidate everything into a single container - ExecutionContext. All methods receive it and when new information is needed, we only modify.

+

In [4]:
#Agent usage example
"""agent = Agent(
model=LlmClient(model="claude-haiku-5-5"),
tools=[calculator, search_web],
instructions="You are a helpful assistant"
)
result = await agent.run("What is 1234 * 5678?")"""

'agent = Agent(\nmodel=LlmClient(model="claude-haiku-5-5"),\ntools=[calculator, search_web],\ninstructions="You are a helpful assistant"\n)\nresult = await agent.run("What is 1234 * 5678?")'

In [5]:
import inspect
import json
import uuid
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Callable, Dict, List, Literal, Optional, Type, Union, get_origin

from dotenv import load_dotenv
from litellm import acompletion
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from pydantic import BaseModel, ConfigDict, Field

load_dotenv()

True

In [6]:
"""Tool schema helpers from chapter 3, extended for the agent: skip the injected context param, support Pydantic models"""
def function_to_input_schema(func) -> dict:
    type_map = {
        str: "string",
        int: "integer",
        float: "number",
        bool: "boolean",
        list: "array",
        dict: "object",
        type(None): "null",
    }
    try:
        signature = inspect.signature(func)
    except ValueError as e:
        raise ValueError(
            f"Failed to get signature for function {func.__name__}: {e}"
        )

    parameters = {} #hold info for each param in function
    for param in signature.parameters.values():
        if param.name in ("self", "context"):  # context is injected by FunctionTool, not supplied by the LLM
            continue
        annotation = param.annotation
        if isinstance(annotation, type) and issubclass(annotation, BaseModel):
            parameters[param.name] = annotation.model_json_schema()  # Pydantic model -> full JSON schema
        else:
            base_type = get_origin(annotation) or annotation  # list[str] -> list
            parameters[param.name] = {"type": type_map.get(base_type, "string")}

    required = [
        param.name
        for param in signature.parameters.values()
        if param.default == inspect._empty and param.name not in ("self", "context")
    ]

    return {
        "type": "object",
        "properties": parameters,
        "required": required,
    }


#wrap input schema with funct name and docstr to create tool def
def format_tool_definition(name: str, description: str, parameters: dict) -> dict:
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": parameters,
        },
    }


#keep the text blocks of an MCP tool result
def _extract_text_content(result) -> str:
    return "\n".join(block.text for block in result.content if block.type == "text")

Core Agent Data Types

In [7]:
class Message(BaseModel):
    """A text message in the conversation."""
    type: Literal["message"] = "message"
    role: Literal["system", "user", "assistant"]
    content: str

class ToolCall(BaseModel):
    """LLM's request to execute a tool."""
    type: Literal["tool_call"] = "tool_call"
    tool_call_id: str
    name: str
    arguments: dict

class ToolResult(BaseModel):
    """Result from tool execution."""
    type: Literal["tool_result"] = "tool_result"
    tool_call_id: str
    name: str
    status: Literal["success", "error"]
    content: list

ContentItem = Union[Message, ToolCall, ToolResult]


"""Event class letting each occurrence (user msg, tool cal, tool result) be wrapped with author and timestamp metadata."""
class Event(BaseModel):
    """A recorded occurrence during agent execution."""
    id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    execution_id: str
    timestamp: float = Field(default_factory=lambda: datetime.now().timestamp())
    author: str  # "user" or agent name
    content: List[ContentItem] = Field(default_factory=list)


"""ExecutionContext dataclass gathering agent's state - events, execution ID, step count, custom state and final result."""
@dataclass
class ExecutionContext:
    """Central storage for all execution state."""
    
    execution_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    events: List[Event] = field(default_factory=list)
    current_step: int = 0
    state: Dict[str, Any] = field(default_factory=dict)
    final_result: Optional[str | BaseModel] = None
    
    def add_event(self, event: Event):
        """Append an event to the execution history."""
        self.events.append(event)
    
    def increment_step(self):
        """Move to the next execution step."""
        self.current_step += 1

Tool Abstraction: BaseTool abstr class which all tools must inherit from specifying necessary metadata and async execution

In [8]:
class BaseTool(ABC):
    """Abstract base class for all tools."""
    
    def __init__(
        self, 
        name: str = None, 
        description: str = None, 
        tool_definition: Dict[str, Any] = None,
    ):
        self.name = name or self.__class__.__name__
        self.description = description or self.__doc__ or ""
        self._tool_definition = tool_definition
    
    @property
    def tool_definition(self) -> Dict[str, Any] | None:
        return self._tool_definition
    
    @abstractmethod
    async def execute(self, context: ExecutionContext, **kwargs) -> Any:
        pass
    
    async def __call__(self, context: ExecutionContext, **kwargs) -> Any:
        return await self.execute(context, **kwargs)


"""FunctionTool wraps any function under the BaseTool interface, checks at runtime if context is needed and properly passes it,
generates input schema from the funct signature"""

class FunctionTool(BaseTool):
    """Wraps a Python function as a BaseTool."""
    
    def __init__(
        self, 
        func: Callable, 
        name: str = None, 
        description: str = None,
        tool_definition: Dict[str, Any] = None
    ):
        self.name = name or func.__name__
        self.description = description or (func.__doc__ or "").strip()
        self.func = func
        self.needs_context = 'context' in inspect.signature(func).parameters
        
        name = name or func.__name__
        description = description or (func.__doc__ or "").strip()
        tool_definition = tool_definition or self._generate_definition()
        
        super().__init__(
            name=name, 
            description=description, 
            tool_definition=tool_definition
        )
    
    async def execute(self, context: ExecutionContext, **kwargs) -> Any:
        """Execute the wrapped function."""
        if self.needs_context:
            result = self.func(context=context, **kwargs)
        else:
            result = self.func(**kwargs)
        
        # Handle both sync and async functions
        if inspect.iscoroutine(result):
            return await result
        return result
    
    def _generate_definition(self) -> Dict[str, Any]:
        """Generate tool definition from function signature."""
        parameters = function_to_input_schema(self.func)
        return format_tool_definition(self.name, self.description, parameters)


"""@tool decorator: shorthand for FunctionTool(func), usable as @tool or @tool(name=..., description=...)"""
def tool(func: Callable = None, *, name: str = None, description: str = None):
    def decorator(f: Callable) -> FunctionTool:
        return FunctionTool(f, name=name, description=description)
    return decorator(func) if func else decorator


"""Load MCP tools"""
async def load_mcp_tools(connection: dict) -> list[BaseTool]:
    """Load tools from an MCP server and convert to FunctionTools."""
    tools = []
    
    async with stdio_client(StdioServerParameters(**connection)) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            mcp_tools = await session.list_tools()
            
            for mcp_tool in mcp_tools.tools:
                func_tool = _create_mcp_tool(mcp_tool, connection)
                tools.append(func_tool)
    
    return tools



def _create_mcp_tool(mcp_tool, connection: dict) -> FunctionTool:
    """Create a FunctionTool that wraps an MCP tool."""
    
    async def call_mcp(**kwargs):
        async with stdio_client(StdioServerParameters(**connection)) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool(mcp_tool.name, kwargs)
                return _extract_text_content(result)
    
    tool_definition = {
        "type": "function",
        "function": {
            "name": mcp_tool.name,
            "description": mcp_tool.description,
            "parameters": mcp_tool.inputSchema,
        }
    }
    
    return FunctionTool(
        func=call_mcp,
        name=mcp_tool.name,
        description=mcp_tool.description,
        tool_definition=tool_definition
    )

LLM  communication layer - build pluggable layer for sending requests to LLM API and receiving standardized responses

In [9]:
class LlmRequest(BaseModel):
    """Request object for LLM calls."""
    model_config = ConfigDict(arbitrary_types_allowed=True)
    instructions: List[str] = Field(default_factory=list)
    contents: List[ContentItem] = Field(default_factory=list)
    tools: List[BaseTool] = Field(default_factory=list)
    tool_choice: Optional[str] = None


class LlmResponse(BaseModel):
    """Response object from LLM calls."""
    content: List[ContentItem] = Field(default_factory=list)
    error_message: Optional[str] = None
    usage_metadata: Dict[str, Any] = Field(default_factory=dict)


class LlmClient:
    """Client for LLM API calls using LiteLLM."""
    
    def __init__(self, model: str, **config):
        self.model = model
        self.config = config
    
    async def generate(self, request: LlmRequest) -> LlmResponse:
        """Generate a response from the LLM."""
        try:
            messages = self._build_messages(request)
            tools = [t.tool_definition for t in request.tools] if request.tools else None
            
            response = await acompletion(
                model=self.model,
                messages=messages,
                tools=tools,
                **({"tool_choice": request.tool_choice} 
                   if request.tool_choice else {}),
                **self.config
            )
            
            return self._parse_response(response)
        except Exception as e:
            return LlmResponse(error_message=str(e))

    def _build_messages(self, request: LlmRequest) -> List[dict]:
        """Convert LlmRequest to API message format."""
        messages = []
        
        for instruction in request.instructions:
            messages.append({"role": "system", "content": instruction})
        
        for item in request.contents:
            if isinstance(item, Message):
                messages.append({"role": item.role, "content": item.content})
                
            elif isinstance(item, ToolCall):
                tool_call_dict = {
                    "id": item.tool_call_id,
                    "type": "function",
                    "function": {
                        "name": item.name,
                        "arguments": json.dumps(item.arguments)
                    }
                }
                # Append to previous assistant message if exists
                if messages and messages[-1]["role"] == "assistant":
                    messages[-1].setdefault("tool_calls", []).append(tool_call_dict)
                else:
                    messages.append({
                        "role": "assistant",
                        "content": None,
                        "tool_calls": [tool_call_dict]
                    })
                    
            elif isinstance(item, ToolResult):
                messages.append({
                    "role": "tool",
                    "tool_call_id": item.tool_call_id,
                    "content": str(item.content[0]) if item.content else ""
                })
        
        return messages

    def _parse_response(self, response) -> LlmResponse:
        """Convert API response to LlmResponse."""
        choice = response.choices[0]
        content_items = []
        
        if choice.message.content:
            content_items.append(Message(
                role="assistant",
                content=choice.message.content
            ))
        
        if choice.message.tool_calls:
            for tc in choice.message.tool_calls:
                content_items.append(ToolCall(
                    tool_call_id=tc.id,
                    name=tc.function.name,
                    arguments=json.loads(tc.function.arguments)
                ))
        
        return LlmResponse(
            content=content_items,
            usage_metadata={
                "input_tokens": response.usage.prompt_tokens,
                "output_tokens": response.usage.completion_tokens,
            }
        )

In [10]:
# Create client
client = LlmClient(model="gpt-5.4-mini")

# Build request
request = LlmRequest(
    instructions=["You are a helpful assistant."],
    contents=[Message(role="user", content="What is 2 + 2?")],
)

# Generate response
response = await client.generate(request)

# Response contains the answer
for item in response.content:
    if isinstance(item, Message):
        print(item.content)  # "4"


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



Agent

In [11]:
"""Bundling agent's final output and context"""
@dataclass
class AgentResult:
    """Result of an agent execution."""
    output: str | BaseModel
    context: ExecutionContext


class Agent:
    def __init__(
        self,
        model: LlmClient,
        tools: List[BaseTool] = None,
        instructions: str = "",
        max_steps: int = 10,
        name: str = "agent",
        output_type: Optional[Type[BaseModel]] = None,  # specify pydantic schema

    ):
        self.model = model
        self.instructions = instructions
        self.max_steps = max_steps
        self.name = name
        self.output_type = output_type
        self.output_tool_name = None  # Will be set if output_type provided
        self.tools = self._setup_tools(tools or [])


    def _setup_tools(self, tools: List[BaseTool]) -> List[BaseTool]:
        if self.output_type is not None:
            @tool(
                name="final_answer",
                description="Return the final structured answer matching the required schema."
            )
            def final_answer(output: self.output_type) -> self.output_type:
                return self.output_type.model_validate(output)  # LLM sends a dict -> validated Pydantic instance
            
            tools = list(tools)  # Create a copy to avoid modifying the original
            tools.append(final_answer)
            self.output_tool_name = "final_answer"
        
        return tools


    """run() in the main entry point: set up the context, loop calling step() until result is reached"""
    async def run(
        self,
        user_input: str,
        context: ExecutionContext = None
    ) -> AgentResult:
        # Create or reuse context
        if context is None:
            context = ExecutionContext()

        # Add user input as the first event in execution history
        user_event = Event(
            execution_id=context.execution_id,
            author="user",
            content=[Message(role="user", content=user_input)]
        )
        context.add_event(user_event)

        # Execute steps until completion or max steps reached
        while not context.final_result and context.current_step < self.max_steps:
            await self.step(context)

            # Check if the last event is a final response
            last_event = context.events[-1]
            if self._is_final_response(last_event):
                context.final_result = self._extract_final_result(last_event)

        return AgentResult(output=context.final_result, context=context)


    """Helpers to determine when the agent is done and how to pull the answer"""
    def _is_final_response(self, event: Event) -> bool:
        if self.output_tool_name:
            # For structured output: check if final_answer tool succeeded
            for item in event.content:
                if (isinstance(item, ToolResult) 
                    and item.name == self.output_tool_name 
                    and item.status == "success"):
                    return True
            return False
        
        # Free-text responses: final when the LLM answered without calling tools
        has_tool_calls = any(isinstance(c, ToolCall) for c in event.content)
        has_tool_results = any(isinstance(c, ToolResult) for c in event.content)
        return not has_tool_calls and not has_tool_results

    def _extract_final_result(self, event: Event) -> str | BaseModel:
        if self.output_tool_name:
            # Extract structured output from final_answer tool result
            for item in event.content:
                if (isinstance(item, ToolResult) 
                    and item.name == self.output_tool_name 
                    and item.status == "success" 
                    and item.content):
                    return item.content[0]
        
        # Free-text responses: return the assistant's message
        for item in event.content:
            if isinstance(item, Message) and item.role == "assistant":
                return item.content
        return None


    """step() the core think-act cycle: prepare request for the LLM, record response, execute tool res, bump step"""
    async def step(self, context: ExecutionContext):
        # Prepare what to send to the LLM
        llm_request = self._prepare_llm_request(context)
        
        # Get LLM's decision
        llm_response = await self.think(llm_request)
        if llm_response.error_message:
            raise RuntimeError(f"LLM request failed: {llm_response.error_message}")
        
        # Record LLM response as an event
        response_event = Event(
            execution_id=context.execution_id,
            author=self.name,
            content=llm_response.content,
        )
        context.add_event(response_event)
        
        # Execute tools if the LLM requested any
        tool_calls = [c for c in llm_response.content if isinstance(c, ToolCall)]
        if tool_calls:
            tool_results = await self.act(context, tool_calls)
            tool_event = Event(
                execution_id=context.execution_id,
                author=self.name,
                content=tool_results,
            )
            context.add_event(tool_event)
        
        context.increment_step()

    """put relevant items from the full context and packages into LlmRequest for the LLM"""
    def _prepare_llm_request(self, context: ExecutionContext) -> LlmRequest:
        # Flatten events into content items
        flat_contents = []
        for event in context.events:
            flat_contents.extend(event.content)

        # Determine tool choice strategy
        if self.output_tool_name:
            tool_choice = "required"  # Force tool usage for structured output
        elif self.tools:
            tool_choice = "auto"
        else:
            tool_choice = None

        #package the extracted data
        return LlmRequest(
            instructions=[self.instructions] if self.instructions else [],
            contents=flat_contents,
            tools=self.tools,
            tool_choice=tool_choice,
        )

    """wrapper to call the LLM using the prepared request and return standardized response"""
    async def think(self, llm_request: LlmRequest) -> LlmResponse:
        return await self.model.generate(llm_request) #access to the model's generate method

    """executes the list of tool calls generated by LLM, handling success and failure"""
    async def act(self,context: ExecutionContext,tool_calls: List[ToolCall]) -> List[ToolResult]:
        tools_dict = {tool.name: tool for tool in self.tools}
        results = []

        for tool_call in tool_calls:
            if tool_call.name not in tools_dict:
                results.append(ToolResult(
                    tool_call_id=tool_call.tool_call_id,
                    name=tool_call.name,
                    status="error",
                    content=[f"Tool '{tool_call.name}' not found"],
                ))
                continue

            tool = tools_dict[tool_call.name]

            try:
                output = await tool(context, **tool_call.arguments)
                results.append(ToolResult(
                    tool_call_id=tool_call.tool_call_id,
                    name=tool_call.name,
                    status="success",
                    content=[output],
                ))
            except Exception as e:
                results.append(ToolResult(
                    tool_call_id=tool_call.tool_call_id,
                    name=tool_call.name,
                    status="error",
                    content=[str(e)],
                ))

        return results


"""SentimentAnalysis Pydantic model and use structured output to get guaranteed sentiment, confidence, key phrases from agent"""

class SentimentAnalysis(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float
    key_phrases: List[str]

agent = Agent(
    model=LlmClient(model="claude-haiku-5-5"),
    tools=[],
    instructions="Analyze the sentiment of the provided text.",
    output_type=SentimentAnalysis
)

result = await agent.run("This product exceeded my expectations! Highly recommend.")
print(f"Sentiment: {result.output.sentiment}")      # "positive"
print(f"Confidence: {result.output.confidence}")    # 0.92
print(f"Key phrases: {result.output.key_phrases}")  # ["exceeded expectations", "highly recommend"]

Sentiment: positive
Confidence: 0.97
Key phrases: ['exceeded my expectations', 'Highly recommend']


In [ ]:
import os
from datasets import load_dataset
from scratch_agents.eval.gaia import GaiaOutput, gaia_prompt, run_experiment
from scratch_agents.tools import load_mcp_tools

level1_problems = load_dataset("gaia-benchmark/GAIA", "2023_level1", split="validation")
problems = level1_problems.select(range(10))

tavily_connection = {
    "command": "npx",
    "args": ["-y", "tavily-mcp@latest"],
    "env": {"TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")}
}

mcp_tools = await load_mcp_tools(tavily_connection)

NameError: name 'problems' is not defined